# GRPO on NFL spread picks
Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all. Approve the Google Drive pop-up when it appears: every result is written to Drive as it is produced, so a Colab disconnect loses nothing. If the session dies, just Run all again and it picks up from the last checkpoint.

Change `REWARD` to `"brier"` for the second experiment.

In [ ]:
REPO_URL = "https://github.com/ZekeRomero/nfl-grpo.git"
MODEL    = "unsloth/Qwen2.5-1.5B-Instruct"
REWARD   = "win"        # "win" or "brier"   <- the only line you change between the two experiments
SMOKE_TEST = False      # True = quick end-to-end check. False = the real run.

MAX_STEPS = 300          # training steps; each step = 1 game x NUM_GEN sampled answers
NUM_GEN   = 8            # answers sampled per game. Leave at 8.
MAX_NEW_TOKENS = 400     # length limit per answer
EVAL_GAMES = None        # None = every validation/test game
META_N     = 150         # games in the reworded-prompt check
SEED = 0
SAVE_TO_DRIVE = True

if SMOKE_TEST:
    MAX_STEPS, EVAL_GAMES, META_N = 10, 40, 10

In [ ]:
import os
RUN = f"smoke_{REWARD}" if SMOKE_TEST else f"{REWARD}_{MAX_STEPS}steps"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/nfl-grpo-runs/{RUN}"
else:
    OUT = f"/content/runs/{RUN}"
os.makedirs(OUT, exist_ok=True)
print("Saving everything to", OUT)

In [ ]:
!pip install -q unsloth
!rm -rf /content/nfl-grpo
!git clone -q $REPO_URL /content/nfl-grpo
%cd /content/nfl-grpo
!python scripts/build_dataset.py
!python scripts/run_baselines.py
!cp data/baselines_*.csv "$OUT"/

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
import glob, json, torch, pandas as pd
from datasets import Dataset
from nflgrpo.data import split
from nflgrpo.prompts import VARIANTS, messages
from nflgrpo.rewards import REWARD_SETS, home_pick_monitor, parse_output
from nflgrpo.evaluate import score, consistency, table

model, tokenizer = FastLanguageModel.from_pretrained(MODEL, max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, random_state=SEED,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth")
tokenizer.padding_side = "left"

df = pd.read_csv("data/games_features.csv")
train_df, val_df, test_df = split(df)
train_ds = Dataset.from_list([json.loads(l) for l in open("data/train.jsonl")]).shuffle(seed=SEED)
print(len(train_ds), "training games")

## Evaluation helpers
Greedy decoding, so a given model always gives the same pick for the same prompt.

In [ ]:
@torch.no_grad()
def generate(msg_lists, batch_size=16):
    outs = []
    for i in range(0, len(msg_lists), batch_size):
        texts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                 for m in msg_lists[i:i+batch_size]]
        enc = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        outs += tokenizer.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return outs

def evaluate(part, tag, **prompt_kw):
    part = part.head(EVAL_GAMES) if EVAL_GAMES else part
    texts = generate([messages(r, **prompt_kw) for _, r in part.iterrows()])
    parsed = [parse_output(t) for t in texts]
    pd.DataFrame({"game_id": part.game_id.values, "label": part.label.values, "output": texts,
                  "pick": [p[0] for p in parsed], "conf": [p[1] for p in parsed]}
                 ).to_json(f"{OUT}/outputs_{tag}.jsonl", orient="records", lines=True)
    return score(part, [p[0] for p in parsed], [p[1] for p in parsed])

def metamorphic(part, tag):
    """Same game, reworded prompt: the pick should not change."""
    part = part.sample(min(META_N, len(part)), random_state=SEED)
    picks = {k: [parse_output(t)[0] for t in generate([messages(r, **kw) for _, r in part.iterrows()])]
             for k, kw in VARIANTS.items()}
    pd.DataFrame(picks, index=part.game_id.values).to_csv(f"{OUT}/metamorphic_picks_{tag}.csv")
    res = consistency(picks)
    json.dump(res, open(f"{OUT}/consistency_{tag}.json", "w"))
    return res

## Format check
Prints what the untrained model actually writes on 24 games, and how many answers contain a usable pick.

In [ ]:
FastLanguageModel.for_inference(model)
_s = val_df.sample(24, random_state=SEED)
_t = generate([messages(r) for _, r in _s.iterrows()])
_p = [parse_output(t) for t in _t]
pd.DataFrame({"output": _t, "pick": [p[0] for p in _p], "conf": [p[1] for p in _p]}
             ).to_json(f"{OUT}/format_check.jsonl", orient="records", lines=True)
pick_rate = sum(p[0] is not None for p in _p) / len(_p)
valid_rate = sum(p[2] for p in _p) / len(_p)
print(f"usable pick: {pick_rate:.0%}   fully valid: {valid_rate:.0%}\n")
for t in _t[:3]:
    print(t, "\n" + "-" * 60)
if pick_rate < 0.5:
    print("NOTE: fewer than half of the untrained answers contain a usable pick. The run continues: "
          "training normally fixes the format within about 100 steps. Untrained win rates will rest on few bets.")

## Before training
Skipped automatically if a previous session already saved these results.

In [ ]:
before_path = f"{OUT}/results_before.csv"
if os.path.exists(before_path) and os.path.exists(f"{OUT}/consistency_before.json"):
    print("Found saved before-training results, not recomputing.")
    before = pd.read_csv(before_path, index_col=0)
    meta_before = json.load(open(f"{OUT}/consistency_before.json"))
else:
    FastLanguageModel.for_inference(model)
    r = {"untrained_val": evaluate(val_df, "untrained_val"),
         "untrained_test": evaluate(test_df, "untrained_test")}
    meta_before = metamorphic(test_df, "before")
    before = table(r).round(2); before.to_csv(before_path)
display(before); print(meta_before)

## GRPO training
Watch `reward` and the per-function rewards in the log. The format reward should climb fast; the outcome reward will be very noisy by design. A checkpoint is written to Drive every 50 steps.

In [ ]:
from trl import GRPOConfig, GRPOTrainer
FastLanguageModel.for_training(model)
funcs = REWARD_SETS[REWARD] + [home_pick_monitor]   # last one is logged only, weight 0
args = GRPOConfig(
    reward_weights=[1.0] * len(REWARD_SETS[REWARD]) + [0.0],
    output_dir=f"{OUT}/checkpoints", seed=SEED, report_to="none",
    learning_rate=5e-6, lr_scheduler_type="cosine", warmup_ratio=0.1, weight_decay=0.01,
    num_generations=NUM_GEN, per_device_train_batch_size=NUM_GEN, gradient_accumulation_steps=1,
    max_completion_length=MAX_NEW_TOKENS, temperature=1.0,
    max_steps=MAX_STEPS, logging_steps=5, save_steps=50, save_total_limit=2,
)
trainer = GRPOTrainer(model=model, processing_class=tokenizer, args=args,
                      reward_funcs=funcs, train_dataset=train_ds)
resume = bool(glob.glob(f"{OUT}/checkpoints/checkpoint-*"))
print("Resuming from the last checkpoint." if resume else "Starting training from step 0.")
trainer.train(resume_from_checkpoint=resume)
pd.DataFrame(trainer.state.log_history).to_csv(f"{OUT}/train_log.csv", index=False)
model.save_pretrained(f"{OUT}/lora"); tokenizer.save_pretrained(f"{OUT}/lora")

## After training

In [ ]:
FastLanguageModel.for_inference(model)
r = {f"grpo_{REWARD}_val": evaluate(val_df, f"grpo_{REWARD}_val"),
     f"grpo_{REWARD}_test": evaluate(test_df, f"grpo_{REWARD}_test")}
meta_after = metamorphic(test_df, "after")
results = pd.concat([before, table(r).round(2)]); results.to_csv(f"{OUT}/results.csv")
display(results)
print("consistency before:", meta_before); print("consistency after: ", meta_after)
print("\nDone. All files are in:", OUT)